In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

# Configuração  Apache Iceberg
spark = SparkSession.builder \
    .appName("Trabalho_Iceberg_Exclusivo") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "warehouse_iceberg") \
    .getOrCreate()

print("Apache Iceberg configurado ")

26/04/27 00:02:33 WARN Utils: Your hostname, MSI resolves to a loopback address: 127.0.1.1; using 172.25.167.157 instead (on interface eth0)
26/04/27 00:02:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/elison/trabalho_eng_dados/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/elison/.ivy2/cache
The jars for the packages stored in: /home/elison/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9738f41e-5245-470e-a39b-41ea291579ea;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 in central
:: resolution report :: resolve 132ms :: artifacts dl 7ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.sp

Apache Iceberg configurado 


In [2]:
spark.sql("CREATE DATABASE IF NOT EXISTS local.db_projeto")

DataFrame[]

In [3]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.db_projeto.tabela_jogos (
        id bigint,
        nome string,
        genero string,
        ano int
    ) 
    USING iceberg
""")

DataFrame[]

In [4]:
spark.sql("""
    INSERT INTO local.db_projeto.tabela_jogos VALUES
    (1, 'FIFA 23', 'Esporte', 2023),
    (2, 'God of War', 'Ação', 2022),
    (3, 'Minecraft', 'Sandbox', 2011)
""")
print(" Dados inseridos ")

 Dados inseridos 


In [5]:
# Mudando o gênero do Minecraft para 'Construção'
spark.sql("""
    UPDATE local.db_projeto.tabela_jogos 
    SET genero = 'Construção' 
    WHERE id = 3
""")
print("Update realizado ")

Update realizado 


In [6]:
# Removendo o jogo FIFA (ID 1)
spark.sql("DELETE FROM local.db_projeto.tabela_jogos WHERE id = 1")
print(" Delete realizado")

 Delete realizado


In [11]:
spark.sql("SELECT * FROM local.db_projeto.tabela_jogos").show()

+---+----------+----------+----+
| id|      nome|    genero| ano|
+---+----------+----------+----+
|  3| Minecraft|Construção|2011|
|  2|God of War|      Ação|2022|
+---+----------+----------+----+



In [10]:
#excluir duplicatas
df_limpo = spark.table("local.db_projeto.tabela_jogos").dropDuplicates()
df_limpo.writeTo("local.db_projeto.tabela_jogos").overwritePartitions()

In [13]:
# Mostra o ID da versão, quando foi feita e que tipo de operação foi (append, delete, overwrite)
spark.sql("""
    SELECT committed_at, snapshot_id, operation 
    FROM local.db_projeto.tabela_jogos.snapshots 
    ORDER BY committed_at ASC
""").show(truncate=False)

+-----------------------+-------------------+---------+
|committed_at           |snapshot_id        |operation|
+-----------------------+-------------------+---------+
|2026-04-26 23:56:10.658|1086654248999677675|append   |
|2026-04-26 23:56:18.486|8547521158190558413|append   |
|2026-04-26 23:59:09.334|7726632082124737430|overwrite|
|2026-04-26 23:59:41.497|5368081661990145789|delete   |
|2026-04-27 00:01:57.503|5481359258557187899|delete   |
|2026-04-27 00:02:07.05 |1313256589711676522|delete   |
|2026-04-27 00:03:26.029|1551280330427088984|append   |
|2026-04-27 00:03:31.253|9020572852767687712|overwrite|
|2026-04-27 00:03:33.864|3065037218816266463|delete   |
|2026-04-27 00:04:22.574|2870272070992758684|overwrite|
|2026-04-27 00:04:34.265|1168254166440548900|overwrite|
+-----------------------+-------------------+---------+



In [14]:
# Verificar histórico
spark.sql("SELECT * FROM local.db_projeto.tabela_jogos.history").show()

+--------------------+-------------------+-------------------+-------------------+
|     made_current_at|        snapshot_id|          parent_id|is_current_ancestor|
+--------------------+-------------------+-------------------+-------------------+
|2026-04-26 23:56:...|1086654248999677675|               NULL|               true|
|2026-04-26 23:56:...|8547521158190558413|1086654248999677675|               true|
|2026-04-26 23:59:...|7726632082124737430|8547521158190558413|               true|
|2026-04-26 23:59:...|5368081661990145789|7726632082124737430|               true|
|2026-04-27 00:01:...|5481359258557187899|5368081661990145789|               true|
|2026-04-27 00:02:...|1313256589711676522|5481359258557187899|               true|
|2026-04-27 00:03:...|1551280330427088984|1313256589711676522|               true|
|2026-04-27 00:03:...|9020572852767687712|1551280330427088984|               true|
|2026-04-27 00:03:...|3065037218816266463|9020572852767687712|               true|
|202

In [18]:
# Substituir o número abaixo por um snapshot_id do  histórico
id_antigo = 1551280330427088984
df_passado = spark.read \
    .option("snapshot-id", id_antigo) \
    .table("local.db_projeto.tabela_jogos")

print(" Lendo a tabela (com duplicadas):")
df_passado.show()

 Lendo a tabela (com duplicadas):
+---+----------+----------+----+
| id|      nome|    genero| ano|
+---+----------+----------+----+
|  1|   FIFA 23|   Esporte|2023|
|  2|God of War|      Ação|2022|
|  3| Minecraft|   Sandbox|2011|
|  2|God of War|      Ação|2022|
|  2|God of War|      Ação|2022|
|  3| Minecraft|Construção|2011|
|  3| Minecraft|Construção|2011|
+---+----------+----------+----+

